In [1]:
import pandas as pd   
import numpy as np    
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

import seaborn as sns
import matplotlib.pyplot as plt


### 데이터 불러오기

In [2]:
df = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("submission_example.csv")

### 데이터 설명

| 변수명           | 데이터타입 | 설명                        | 범주 값 또는 예시                                                                                                                         |
| ------------- | ----- | ------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **age**       | 수치형   | 나이                        | -                                                                                                                                  |
| **job**       | 범주형   | 직업                        | admin, unknown, unemployed, management, housemaid, entrepreneur, student, blue-collar, self-employed, retired, technician, service |
| **marital**   | 범주형   | 결혼상태                      | married, divorced,single (사별해도 divorced로 포함)                                                                                                          |
| **education** | 범주형   | 학력                        | unknown, secondary, primary, tertiary                                                                                              |
| **default**   | 범주형   | 신용 채무 불이행 여부              | yes, no                                                                                                                            |
| **balance**   | 수치형   | 연평균 잔액 (단위: 유로)           | -                                                                                                                                  |
| **housing**   | 범주형   | 주택 대출 여부                  | yes, no                                                                                                                            |
| **loan**      | 범주형   | 개인 대출 여부                  | yes, no                                                                                                                            |
| **contact**   | 범주형   | 최근 연락 수단                  | unknown, telephone, cellular                                                                                                       |
| **day**       | 수치형   | 최근 연락 날짜                  | 1 \~ 31                                                                                                                            |
| **month**     | 범주형   | 최근 연락한 달                  | jan, feb, mar, ..., dec                                                                                                            |
| **duration**  | 수치형   | 최근 통화 시간 (초)              | -                                                                                                                                  |
| **campaign**  | 수치형   | 이번 캠페인 중 고객에게 연락한 횟수      | -                                                                                                                                  |
| **pdays**     | 수치형   | 이전 캠페인 이후 경과일 (-1은 연락 없음) | -                                                                                                                                  |
| **previous**  | 수치형   | 이전 캠페인에서 고객에게 연락한 횟수      | -                                                                                                                                  |
| **poutcome**  | 범주형   | 이전 캠페인 결과                 | unknown, other, failure, success                                                                                                   |
| **label**     | 범주형   | 정기예금 가입 여부 (타겟 변수)        | 1 = 가입, 0 = 미가입                                                                                                                    |


## EDA

### 결측치 확인

#### Target

- 범주형 비율 해석

    - job
        - student, retired, unemployed : 양수 -> 가입자 비중이 더 높음
        - blue-collar, services : 음수 -> 마케팅 효율이 낮은 직업에 해당

    - education
        - tertiary  > 양수에 해당하므로 고학력자일 수록 정기예금에 관심이 많다는 것을 해석할 수 있음
    
    - contact
        - cellular : 휴대폰에 해당하는데 이로 연락을 했을 때 가입할 확률이 올라감
    
    - month
        - mar apr, oct, sep : 양수 
        - may : significantly : 음수

    - martial
        - single : 양수 
            -> 독신일 경우 독립적이기에 가입가능성이 올라가는 것으로 해석 가능
        
        - married : 음수 

    - poutcome 
        - success : 이전 캠페인시 성공시 가입확률 올라감
        - unknown : 과거 기록이 없을 때는 음수일 경우가 많기에 주의

    - default, loan, housing
        - 신용 불이행 X, 대출이 없을 수록 가입확률이 높아짐


## 수치형 변수

### train_test data 나누기

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['label' , 'ID'])  
y = df['label']               
num_col = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
cat_col = ['month','job','education','poutcome', 'default','loan', 'housing','marital','contact','day']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## PipeLine

#### 범주형 변수 pipeline 작성

- one hot encoder : binary 변수 일때 사용
    - 'default', 'loan', 'housing' (Y/N) , 'marital', 'contact' (유의미한 most_frequent에 해당하는 범주가 있음)
- target encder : 범주와 타겟간의 연관성이 명확할 때 사용
    - job, poutcome,
- ordinal encoder : 순서 있는 범주에 사용
    - education

In [4]:
#!pip install category_encoders 

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer
import numpy as np

# 월 -> 숫자 변환
month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
             'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}

education_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[['primary', 'secondary', 'tertiary']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

binary_col = ['default', 'loan', 'housing']
binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore'))
])

other_col = ['marital', 'contact']
other_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

def add_paseng(df):
    df = df.copy()
    df['month_num'] = df['month'].map(month_map).astype(int)
    df['day_of_year'] = df['day'] + (df['month_num'] - 1) * 30
    df['poutcome_success'] = (df['poutcome']=='success').astype(int)
    return df[['day_of_year','poutcome_success']]

day_poutcome_transformer = Pipeline([
    ('adder', FunctionTransformer(add_paseng, validate=False))
])

job_edu_transformer = Pipeline([
    ('concat', FunctionTransformer(
        lambda df: (df['job'] + '_' + df['education']).to_frame(name='job_education'),
        validate=False
    )),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

log_transformer = FunctionTransformer(np.log1p, validate=True)


#pdays 에 대해서는 -1 (연락없음)과 같은 특수 카테고리기 때문에 -1 변수를 999로 할당함.
pdays_transformer = Pipeline([
    ('minus1_to_nan', FunctionTransformer(lambda x: x.replace(-1, np.nan))),
    ('imputer', SimpleImputer(strategy='constant', fill_value=999)),
])

day_pout_pipeline = Pipeline([
    ('adder', FunctionTransformer(add_paseng, validate=False))
])


In [50]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.compose import ColumnTransformer


num_preprocessor = ColumnTransformer(
    transformers=[
        ('age',      StandardScaler(),         ['age']),
        ('balance',  Pipeline([('log', log_transformer)]), ['balance']),
        ('duration', Pipeline([('log', log_transformer)]), ['duration']),
        ('campaign', RobustScaler(),           ['campaign']),
        ('previous','passthrough',             ['previous']),
        ('pdays',    pdays_transformer,        ['pdays']),
    ],
    remainder='drop',
    verbose_feature_names_out=True # ← 그리고 여기!
)

num_preprocessor.set_output(transform='pandas')

ColumnTransformer(transformers=[('age', StandardScaler(), ['age']),
                                ('balance',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>,
                                                                      validate=True))]),
                                 ['balance']),
                                ('duration',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>,
                                                                      validate=True))]),
                                 ['duration']),
                                ('campaign', RobustScaler(), ['campaign']),
                                ('previous', 'passthrough', ['previous']),
                                ('pdays',
                                 Pipeline(steps=[('minus1_to_nan',
                                                  FunctionTransformer(func=<function <lambda> at 0x15ce3fec0>)),
                                                 ('imputer',
                                                  SimpleImputer(fill_value=999,
                                                                strategy='constant'))]),
                                 ['pdays'])])

In [51]:
cat_preprocessor = ColumnTransformer(
    transformers=[
        ('job_edu',   job_edu_transformer,     ['job', 'education']),
        ('education', education_transformer,   ['education']),
        ('binary',    binary_transformer,      binary_col),
        ('other',     other_transformer,       other_col),
        ('day_pout', day_pout_pipeline, ['day','month','poutcome'])
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

cat_preprocessor.set_output(transform='pandas')

ColumnTransformer(transformers=[('job_edu',
                                 Pipeline(steps=[('concat',
                                                  FunctionTransformer(func=<function <lambda> at 0x15ce3fd80>)),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['job', 'education']),
                                ('education',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['primary',
                                                                              's...
                                 ['default', 'loan', 'housing']),
                                ('other',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['marital', 'contact']),
                                ('day_pout',
                                 Pipeline(steps=[('adder',
                                                  FunctionTransformer(func=<function add_paseng at 0x15ce3fe20>))]),
                                 ['day', 'month', 'poutcome'])],
                  verbose_feature_names_out=False)

#### 모델 num_preprocessor

In [52]:
model_pipeline = ColumnTransformer(
    transformers=[
        ('num', num_preprocessor, num_col),
        ('cat', cat_preprocessor, cat_col)
        ],
    remainder='drop',
    verbose_feature_names_out=True)

model_pipeline.set_output(transform='pandas')

ColumnTransformer(transformers=[('num',
                                 ColumnTransformer(transformers=[('age',
                                                                  StandardScaler(),
                                                                  ['age']),
                                                                 ('balance',
                                                                  Pipeline(steps=[('log',
                                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                       validate=True))]),
                                                                  ['balance']),
                                                                 ('duration',
                                                                  Pipeline(steps=[('log',
                                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                       validate=True))]),
                                                                  ['duration']),
                                                                 ('campaign',
                                                                  RobustScaler(),
                                                                  ['campa...
                                                                                  ('onehot',
                                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                                 sparse_output=False))]),
                                                                  ['marital',
                                                                   'contact']),
                                                                 ('day_pout',
                                                                  Pipeline(steps=[('adder',
                                                                                   FunctionTransformer(func=<function add_paseng at 0x15ce3fe20>))]),
                                                                  ['day',
                                                                   'month',
                                                                   'poutcome'])],
                                                   verbose_feature_names_out=False),
                                 ['month', 'job', 'education', 'poutcome',
                                  'default', 'loan', 'housing', 'marital',
                                  'contact', 'day'])])

In [9]:
#Log-transform F1: 0.5426829268292683 >> 로그 스케일링이 훨씬 좋음~
#Log-transform F1: 0.5958041958041959 >> 로그 스케일링 + verbose =-1 + class_weight : balanced

In [18]:
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
import lightgbm as lgbm
from lightgbm import LGBMClassifier  
from sklearn.feature_selection import SequentialFeatureSelector



In [19]:
base_model = lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')

In [21]:
from lightgbm import LGBMClassifier

# 1) 모델 선언
clf = LGBMClassifier(random_state=42, class_weight='balanced')

# 2) 전처리 + 학습
X_trn_prep = model_pipeline.transform(X_train)  # DataFrame
clf.fit(X_trn_prep, y_train)

# 3) 중요도 추출
importances = clf.feature_importances_
features    = X_trn_prep.columns

# 4) 정리
import pandas as pd
fi_df = pd.DataFrame({
    'feature':    features,
    'importance': importances
}).sort_values('importance', ascending=False)

print(fi_df.head(20))


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

[LightGBM] [Info] Number of positive: 2962, number of negative: 22355
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000759 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1258
[LightGBM] [Info] Number of data points in the train set: 25317, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
                                feature  importance
64                          day_of_year         855
2                                     2         493
1                                     1         362
0                                     0         282
5                                     5         172
3                                     3         125
4                                     4          80
57                          h

## Modeling

In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb

In [32]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier
import numpy as np

# 1) Pipeline 정의: 전처리 → 분류기
tune_pipe = Pipeline([
    ('preproc', model_pipeline),     # 이미 짜 둔 ColumnTransformer
    ('clf',     LGBMClassifier(verbose=-1, random_state=42))
])

param_dist = {
    'clf__n_estimators':    np.arange(100,1001,100),
    'clf__learning_rate':   np.arange(0.01,0.31,0.05),
    'clf__max_depth':       np.arange(3,11),
    'clf__subsample':       np.linspace(0.5,1.0,6),
    'clf__colsample_bytree':np.linspace(0.5,1.0,6),
    'clf__class_weight':    ['balanced'] + [{0:1,1:i} for i in range(2,6)],
    'clf__random_state':    [42]
}

# 3) RandomizedSearchCV 설정
random_search = RandomizedSearchCV(
    estimator          = tune_pipe,
    param_distributions= param_dist,
    n_iter             = 50,
    scoring            = 'f1',
    cv                 = 5,
    n_jobs             = -1,
    random_state       = 42,
    refit              = True
)


In [33]:
# 4) 튜닝 실행 (전처리까지 자동 적용)
random_search.fit(X_train, y_train)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preproc',
                                              ColumnTransformer(transformers=[('num',
                                                                               ColumnTransformer(transformers=[('age',
                                                                                                                StandardScaler(),
                                                                                                                ['age']),
                                                                                                               ('balance',
                                                                                                                Pipeline(steps=[('log',
                                                                                                                                 FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                                                                     validate=True))]),
                                                                                                                ['balance']),
                                                                                                               ('duration',
                                                                                                                Pipeline(steps=[('log',
                                                                                                                                 FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                                                                     valid...
                                                              {0: 1, 1: 4},
                                                              {0: 1, 1: 5}],
                                        'clf__colsample_bytree': array([0.5, 0.6, 0.7, 0.8, 0.9, 1. ]),
                                        'clf__learning_rate': array([0.01, 0.06, 0.11, 0.16, 0.21, 0.26]),
                                        'clf__max_depth': array([ 3,  4,  5,  6,  7,  8,  9, 10]),
                                        'clf__n_estimators': array([ 100,  200,  300,  400,  500,  600,  700,  800,  900, 1000]),
                                        'clf__random_state': [42],
                                        'clf__subsample': array([0.5, 0.6, 0.7, 0.8, 0.9, 1. ])},
                   random_state=42, scoring='f1')

In [34]:
# 5) 결과 확인
print("▶ 최적 파라미터:", random_search.best_params_)
print("▶ 검증 F1 (refit 된 모델):",
      f1_score(y_val, random_search.predict(X_val)))

▶ 최적 파라미터: {'clf__subsample': np.float64(1.0), 'clf__random_state': 42, 'clf__n_estimators': np.int64(400), 'clf__max_depth': np.int64(4), 'clf__learning_rate': np.float64(0.060000000000000005), 'clf__colsample_bytree': np.float64(1.0), 'clf__class_weight': {0: 1, 1: 4}}
▶ 검증 F1 (refit 된 모델): 0.6179716237519706


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

In [35]:
best_model = random_search.best_estimator_
print("▶ 최적 파라미터:", random_search.best_params_)

▶ 최적 파라미터: {'clf__subsample': np.float64(1.0), 'clf__random_state': 42, 'clf__n_estimators': np.int64(400), 'clf__max_depth': np.int64(4), 'clf__learning_rate': np.float64(0.060000000000000005), 'clf__colsample_bytree': np.float64(1.0), 'clf__class_weight': {0: 1, 1: 4}}


In [36]:
# 5) 검증셋에 대한 예측 확률 계산
proba_val = best_model.predict_proba(X_val)[:,1]

# 6) Precision-Recall 곡선과 임계값 계산
precision, recall, thresholds = precision_recall_curve(y_val, proba_val)

# 7) F1 스코어가 최대가 되는 threshold 찾기
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx   = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

In [37]:
print(f"▶ 최적 임계값(threshold): {best_threshold:.4f}")
print(f"▶ 해당 시점 F1 스코어: {best_f1:.4f}")

▶ 최적 임계값(threshold): 0.5638
▶ 해당 시점 F1 스코어: 0.6213


In [ ]:
#▶ 최적 임계값(threshold): 0.5638
#▶ 해당 시점 F1 스코어: 0.6213

### 앙상블

In [53]:
from sklearn.ensemble       import StackingClassifier, RandomForestClassifier
from sklearn.linear_model  import LogisticRegression
from lightgbm              import LGBMClassifier
from sklearn.pipeline      import Pipeline

best_lgbm = random_search.best_estimator_.named_steps['clf']


In [55]:
from sklearn.ensemble import RandomForestClassifier

estimators = [
    ('lgbm', best_lgbm),
    ('rf', RandomForestClassifier(
        n_estimators=200,
        max_depth=7,
        class_weight='balanced',
        random_state=42
    ))
]


In [58]:
# 메타 모델은 로지스틱 회귀
stack = StackingClassifier(
    estimators      = estimators,
    final_estimator = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    cv              = 5,
    n_jobs          = -1,
    passthrough     = False
)

# 무한대 → nan, 너무 크면 지정값 → 0 으로 바꾸기
nan_inf_handler = FunctionTransformer(
    lambda X: np.nan_to_num(
        X,
        nan=0.0,        # NaN을 0으로
        posinf=0.0,     # +inf를 0으로
        neginf=0.0      # -inf를 0으로
    ),
    validate=False
)

stack_pipe = Pipeline([
    ('preproc', model_pipeline),
    ('nan_inf', nan_inf_handler),   # ← 여기를 추가
    ('stack',   stack)
])

stack_pipe.fit(X_train, y_train)


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

Pipeline(steps=[('preproc',
                 ColumnTransformer(transformers=[('num',
                                                  ColumnTransformer(transformers=[('age',
                                                                                   StandardScaler(),
                                                                                   ['age']),
                                                                                  ('balance',
                                                                                   Pipeline(steps=[('log',
                                                                                                    FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                                        validate=True))]),
                                                                                   ['balance']),
                                                                                  ('duration',
                                                                                   Pipeline(steps=[('log',
                                                                                                    FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                                        validate=True))]),
                                                                                   ['duration']),
                                                                                  ('campa...
                                                                learning_rate=np.float64(0.060000000000000005),
                                                                max_depth=np.int64(4),
                                                                n_estimators=np.int64(400),
                                                                random_state=42,
                                                                subsample=np.float64(1.0),
                                                                verbose=-1)),
                                                ('rf',
                                                 RandomForestClassifier(class_weight='balanced',
                                                                        max_depth=7,
                                                                        n_estimators=200,
                                                                        random_state=42))],
                                    final_estimator=LogisticRegression(class_weight='balanced',
                                                                       max_iter=1000,
                                                                       random_state=42),
                                    n_jobs=-1))])

In [59]:
from sklearn.metrics import precision_recall_curve, f1_score

# 1-1) 예측 확률 취득
proba_val = stack_pipe.predict_proba(X_val)[:, 1]

# 1-2) 기본 threshold=0.5 로 이진 예측 및 F1
y_pred_default = (proba_val >= 0.5).astype(int)



기본 threshold=0.5 F1: 0.585886722376973


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

In [60]:
print("기본 threshold=0.5 F1:", f1_score(y_val, y_pred_default))

기본 threshold=0.5 F1: 0.585886722376973


In [61]:
# 2-1) precision-recall 커브 계산
precision, recall, thresholds = precision_recall_curve(y_val, proba_val)

# 2-2) 각 threshold 에서 F1 계산
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

# 2-3) 최고 F1 을 주는 임계값 찾기
best_idx      = f1_scores.argmax()
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

print(f"▶ 최적 threshold: {best_threshold:.4f}")
print(f"▶ 최적 임계값 F1:  {best_f1:.4f}")


▶ 최적 threshold: 0.8120
▶ 최적 임계값 F1:  0.6200


In [63]:
#▶ 최적 threshold: 0.8120
#▶ 최적 임계값 F1:  0.62 0 0

In [68]:
import numpy as np
import pandas as pd
from sklearn.pipeline      import Pipeline
from sklearn.ensemble      import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm          import SVC
from xgboost               import XGBClassifier
from lightgbm             import LGBMClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics        import f1_score

# ──────────────────────────────────────────────────────────────────────────
# 1) 튜닝된 LGBMClassifier 꺼내기 (RandomizedSearchCV 결과물에서)
best_lgbm = random_search.best_estimator_.named_steps['clf']

# 2) 추가적인 1차 모델 정의
estimators = [
    ('lgbm', best_lgbm),
    ('rf',   RandomForestClassifier(
                 n_estimators=200,
                 max_depth=7,
                 class_weight='balanced',
                 random_state=42
             )),
    ('xgb',  XGBClassifier(
                 n_estimators=200,
                 max_depth=5,
                 learning_rate=0.1,
                 scale_pos_weight=1,
                 use_label_encoder=False,
                 eval_metric='logloss',
                 random_state=42
             )),
    ('svc',  SVC(
                 probability=True,
                 C=1.0,
                 kernel='rbf',
                 class_weight='balanced',
                 random_state=42
             ))
]

# 3) StackingClassifier 에 passthrough 적용
stack = StackingClassifier(
    estimators      = estimators,
    final_estimator = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    cv              = 5,
    n_jobs          = -1,
    passthrough     = True   # !!! 원본 전처리 피처도 메타 모델에 함께 전달
)

# 무한대 → nan, 너무 크면 지정값 → 0 으로 바꾸기
nan_inf_handler = FunctionTransformer(
    lambda X: np.nan_to_num(
        X,
        nan=0.0,        # NaN을 0으로
        posinf=0.0,     # +inf를 0으로
        neginf=0.0      # -inf를 0으로
    ),
    validate=False)

# 4) 전체 파이프라인: 전처리 → 스태킹
stack_pipe = Pipeline([
    ('preproc', model_pipeline),
    ('nan', nan_inf_handler),
    ('stack',   stack)
])

# ──────────────────────────────────────────────────────────────────────────
# 5) 메타 모델 하이퍼파라미터 그리드 서치
param_grid = {
    'stack__final_estimator__C':            [0.01, 0.1, 1.0, 10.0],
    'stack__final_estimator__class_weight': ['balanced', {0:1, 1:2}, {0:1,1:3}]
}

grid = GridSearchCV(
    estimator   = stack_pipe,
    param_grid  = param_grid,
    scoring     = 'f1',
    cv          = 3,
    n_jobs      = -1,
    verbose     = 1,
    refit       = True
)

grid.fit(X_train, y_train)


Fitting 3 folds for each of 12 candidates, totalling 36 fits


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preproc',
                                        ColumnTransformer(transformers=[('num',
                                                                         ColumnTransformer(transformers=[('age',
                                                                                                          StandardScaler(),
                                                                                                          ['age']),
                                                                                                         ('balance',
                                                                                                          Pipeline(steps=[('log',
                                                                                                                           FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                                                               validate=True))]),
                                                                                                          ['balance']),
                                                                                                         ('duration',
                                                                                                          Pipeline(steps=[('log',
                                                                                                                           FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                                                               validate=Tr...
                                                                        SVC(class_weight='balanced',
                                                                            probability=True,
                                                                            random_state=42))],
                                                           final_estimator=LogisticRegression(class_weight='balanced',
                                                                                              max_iter=1000,
                                                                                              random_state=42),
                                                           n_jobs=-1,
                                                           passthrough=True))]),
             n_jobs=-1,
             param_grid={'stack__final_estimator__C': [0.01, 0.1, 1.0, 10.0],
                         'stack__final_estimator__class_weight': ['balanced',
                                                                  {0: 1, 1: 2},
                                                                  {0: 1,
                                                                   1: 3}]},
             scoring='f1', verbose=1)

In [69]:
print("▶ 메타 모델 최적 파라미터:", grid.best_params_)


▶ 메타 모델 최적 파라미터: {'stack__final_estimator__C': 1.0, 'stack__final_estimator__class_weight': {0: 1, 1: 3}}


In [ ]:
#{'stack__final_estimator__C': 1.0, 'stack__final_estimator__class_weight': {0: 1, 1: 3}}

In [70]:
# 6) 교차검증을 통한 성능 비교
print("LGBM 단독 CV F1:",
      cross_val_score(best_lgbm, model_pipeline.transform(X), y,
                      scoring='f1', cv=5).mean())


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

LGBM 단독 CV F1: 0.6261817217997938


In [71]:
print("Stacking CV F1:",
      cross_val_score(grid.best_estimator_, X, y,
                      scoring='f1', cv=5).mean())



/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

Stacking CV F1: 0.6304055906113198


In [72]:
y_val_pred = grid.predict(X_val)
print("▶ 검증 F1 (스태킹):", f1_score(y_val, y_val_pred))

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: divide by zero encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:387: RuntimeWarning: invalid value encountered in log1p
  return func(X, **(kw_args if kw_args else {}))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_

▶ 검증 F1 (스태킹): 0.6152149944873209
